# IMC25 — Exploratory Data Analysis (EDA)

This notebook provides a practical overview of the IMC25 dataset in this repo:

- Dataset/scene/image counts
- Basic integrity checks (missing files, duplicates)
- Image statistics (resolution, aspect ratio, file size)
- Pose label statistics (R/t sanity checks, camera centers)
- Threshold table inspection

Start Jupyter from the repo root so relative paths like `data/train` resolve correctly.


In [ ]:
from __future__ import annotations

import json
import math
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from PIL import Image
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "data").exists() and (p / "src").exists():
            return p
    return start


REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"
CACHE_DIR = REPO_ROOT / "cache"
CACHE_DIR.mkdir(exist_ok=True)

REPO_ROOT


## Load metadata

This repo includes a few different label/image roots:

- `MODE="full"`: uses `data/train_labels.csv` + `data/train` (full labeled train set)
- `MODE="split"`: uses one of the `data/split_info*.json` configs (train/test split from train)


In [ ]:
MODE = "full"  # {"full", "split"}

# Only used when MODE == "split"
SPLIT_INFO_PATH = DATA_DIR / "split_info.json"

THRESHOLDS_CSV = DATA_DIR / "train_thresholds.csv"
SAMPLE_SUBMISSION_CSV = DATA_DIR / "sample_submission.csv"


def _normalize_from_split_info(path_value: str | Path) -> Path:
    """Resolve split_info.json paths even if they were stored as absolute paths."""
    p = Path(path_value)
    if p.exists():
        return p

    # Try relative to repo root.
    candidate = REPO_ROOT / p
    if candidate.exists():
        return candidate

    # Try rewriting absolute paths that contain a /data/ segment.
    s = str(p)
    if "/data/" in s:
        rel = s.split("/data/", 1)[1]
        candidate = REPO_ROOT / "data" / rel
        if candidate.exists():
            return candidate

    raise FileNotFoundError(f"Could not resolve path: {path_value}")


if MODE == "split":
    split = json.loads(SPLIT_INFO_PATH.read_text())
    TRAIN_ROOT = _normalize_from_split_info(split["train_root"])
    TEST_ROOT = _normalize_from_split_info(split["test_root"])
    TRAIN_LABELS_CSV = _normalize_from_split_info(split["train_labels_csv"])
    TEST_LABELS_CSV = _normalize_from_split_info(split["test_labels_csv"])
else:
    TRAIN_ROOT = DATA_DIR / "train"
    TEST_ROOT = DATA_DIR / "test"
    TRAIN_LABELS_CSV = DATA_DIR / "train_labels.csv"
    TEST_LABELS_CSV = None

train_labels = pd.read_csv(TRAIN_LABELS_CSV)
test_labels = pd.read_csv(TEST_LABELS_CSV) if TEST_LABELS_CSV is not None else None

train_thresholds = pd.read_csv(THRESHOLDS_CSV) if THRESHOLDS_CSV.exists() else None
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_CSV) if SAMPLE_SUBMISSION_CSV.exists() else None

print(
    f"MODE={MODE} | train_labels={train_labels.shape} | "
    f"test_labels={(None if test_labels is None else test_labels.shape)}"
)
TRAIN_ROOT, TEST_ROOT


## Dataset overview

Counts by dataset/scene and quick sanity checks on the label tables.

In [ ]:
def add_image_paths(df: pd.DataFrame, images_root: Path) -> pd.DataFrame:
    df = df.copy()
    df["image_path"] = [images_root / ds / img for ds, img in zip(df["dataset"], df["image"], strict=False)]
    df["image_exists"] = [p.exists() for p in df["image_path"]]
    return df


train_df = add_image_paths(train_labels, TRAIN_ROOT)
test_df = add_image_paths(test_labels, TEST_ROOT) if test_labels is not None else None

display(train_df.head())
print(
    "Train integrity:",
    {
        "rows": len(train_df),
        "missing_images": int((~train_df["image_exists"]).sum()),
        "dup_dataset_image": int(train_df.duplicated(["dataset", "image"]).sum()),
        "datasets": int(train_df["dataset"].nunique()),
        "scenes": int(train_df[["dataset", "scene"]].drop_duplicates().shape[0]),
    },
)

if test_df is not None:
    print(
        "Test integrity:",
        {
            "rows": len(test_df),
            "missing_images": int((~test_df["image_exists"]).sum()),
            "dup_dataset_image": int(test_df.duplicated(["dataset", "image"]).sum()),
            "datasets": int(test_df["dataset"].nunique()),
            "scenes": int(test_df[["dataset", "scene"]].drop_duplicates().shape[0]),
        },
    )


In [ ]:
by_dataset = (
    train_df.groupby("dataset")
    .agg(n_images=("image", "size"), n_scenes=("scene", "nunique"), missing=("image_exists", lambda s: int((~s).sum())))
    .reset_index()
    .sort_values("n_images", ascending=False)
)
display(by_dataset)
px.bar(by_dataset, x="dataset", y="n_images", title="Train: images per dataset").show()


In [ ]:
by_scene = (
    train_df.groupby(["dataset", "scene"])
    .agg(n_images=("image", "size"), missing=("image_exists", lambda s: int((~s).sum())))
    .reset_index()
    .sort_values("n_images", ascending=False)
)
display(by_scene.head(50))

top = by_scene.head(40)
px.bar(top, x="scene", y="n_images", color="dataset", title="Train: images per scene (top 40)").show()


## Filesystem check (train)

Compare labeled images vs files on disk under the selected `TRAIN_ROOT`.

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

train_files = [p for p in TRAIN_ROOT.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS]
train_files_set = {(p.relative_to(TRAIN_ROOT).parts[0], p.name) for p in train_files}
train_labels_set = {(ds, img) for ds, img in zip(train_df["dataset"], train_df["image"], strict=False)}

unlabeled = sorted(train_files_set - train_labels_set)
missing_on_disk = sorted(train_labels_set - train_files_set)

print("Train files on disk:", len(train_files_set))
print("Train labels:", len(train_labels_set))
print("Unlabeled files (first 20):", unlabeled[:20])
print("Missing labeled files (first 20):", missing_on_disk[:20])


## Image statistics

Computes per-image resolution + file sizes. This can be cached under `cache/`.

In [ ]:
USE_CACHE = True
IMG_STATS_CACHE = CACHE_DIR / f"eda_image_stats_{MODE}.csv"


def compute_image_stats(df: pd.DataFrame, *, cache_path: Path | None = None) -> pd.DataFrame:
    if USE_CACHE and cache_path is not None and cache_path.exists():
        out = pd.read_csv(cache_path)
        out["image_path"] = out["image_path"].map(Path)
        return out

    rows: list[dict] = []
    for p in tqdm(df["image_path"].tolist(), desc="reading image headers"):
        p = Path(p)
        row = {
            "image_path": p,
            "file_size_bytes": (p.stat().st_size if p.exists() else np.nan),
            "width": np.nan,
            "height": np.nan,
            "mode": None,
            "error": None,
        }
        if p.exists():
            try:
                with Image.open(p) as im:
                    row["width"], row["height"] = im.size
                    row["mode"] = im.mode
            except Exception as e:
                row["error"] = repr(e)
        rows.append(row)

    stats = pd.DataFrame(rows)
    out = df.reset_index(drop=True).copy()
    for col in ["file_size_bytes", "width", "height", "mode", "error"]:
        out[col] = stats[col]

    out["file_size_mb"] = out["file_size_bytes"] / 1e6
    out["aspect_ratio"] = out["width"] / out["height"]
    out["megapixels"] = (out["width"] * out["height"]) / 1e6

    if USE_CACHE and cache_path is not None:
        tmp = out.copy()
        tmp["image_path"] = tmp["image_path"].map(str)
        tmp.to_csv(cache_path, index=False)

    return out


img_stats = compute_image_stats(train_df, cache_path=IMG_STATS_CACHE)
display(img_stats.head())
print(
    {
        "rows": len(img_stats),
        "bad_headers": int(img_stats["error"].notna().sum()),
        "width_range": (float(img_stats["width"].min()), float(img_stats["width"].max())),
        "height_range": (float(img_stats["height"].min()), float(img_stats["height"].max())),
    }
)


In [ ]:
px.histogram(img_stats, x="width", nbins=60, title="Train: image width distribution").show()
px.histogram(img_stats, x="height", nbins=60, title="Train: image height distribution").show()
px.histogram(img_stats, x="aspect_ratio", nbins=80, title="Train: aspect ratio distribution").show()
px.histogram(img_stats, x="file_size_mb", nbins=80, title="Train: file size (MB) distribution").show()


In [ ]:
summary_res = (
    img_stats.groupby("dataset")
    .agg(
        n_images=("image", "size"),
        width_median=("width", "median"),
        height_median=("height", "median"),
        mp_median=("megapixels", "median"),
        file_mb_median=("file_size_mb", "median"),
    )
    .reset_index()
    .sort_values("mp_median", ascending=False)
)
display(summary_res)
px.scatter(
    summary_res,
    x="width_median",
    y="height_median",
    size="n_images",
    color="dataset",
    title="Train: median resolution by dataset",
).show()


### Quick visual check: sample images

Pick a dataset/scene and render a small image grid.

In [ ]:
from IPython.display import display


def show_image_grid(paths: list[Path], *, ncols: int = 6, tile: int = 220) -> None:
    if not paths:
        print("No images to show")
        return

    images = []
    for p in paths:
        try:
            with Image.open(p) as im:
                im = im.convert("RGB")
                im.thumbnail((tile, tile))
                images.append(im.copy())
        except Exception:
            continue

    if not images:
        print("Could not load any images")
        return

    nrows = math.ceil(len(images) / ncols)
    grid = Image.new("RGB", (ncols * tile, nrows * tile), color=(20, 20, 20))
    for i, im in enumerate(images):
        x = (i % ncols) * tile
        y = (i // ncols) * tile
        grid.paste(im, (x, y))
    display(grid)


SAMPLE_DATASET = by_dataset.iloc[0]["dataset"]
SAMPLE_SCENE = None  # e.g. "peach" or "ET"; set to None for all scenes in the dataset
N_SAMPLES = 24

subset = img_stats[img_stats["dataset"] == SAMPLE_DATASET]
if SAMPLE_SCENE is not None:
    subset = subset[subset["scene"] == SAMPLE_SCENE]

paths = subset.sample(min(N_SAMPLES, len(subset)), random_state=SEED)["image_path"].tolist()
print(f"Showing {len(paths)} images from dataset={SAMPLE_DATASET} scene={SAMPLE_SCENE}")
show_image_grid(paths, ncols=6, tile=220)


## Pose labels: parsing + sanity checks

The label CSVs store `rotation_matrix` (9 floats) and `translation_vector` (3 floats) as `;`-separated strings.

We parse them into numeric arrays, then compute derived metrics like camera center (`C = -R^T t`) and basic validity checks.


In [ ]:
_SEP_RE = re.compile(r"[;\s]+")


def _parse_floats(value: str, expected: int, *, label: str) -> np.ndarray:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        raise ValueError(f"Missing {label}")
    text = str(value).strip().strip("[]()")
    parts = [p for p in _SEP_RE.split(text) if p]
    if len(parts) != expected:
        raise ValueError(f"Expected {expected} floats for {label}, got {len(parts)}: {text!r}")
    return np.asarray([float(p) for p in parts], dtype=np.float64)


def parse_rotation_matrix(value: str) -> np.ndarray:
    return _parse_floats(value, 9, label="rotation_matrix").reshape(3, 3)


def parse_translation_vector(value: str) -> np.ndarray:
    return _parse_floats(value, 3, label="translation_vector")


def camera_center_world(R_cw: np.ndarray, t_cw: np.ndarray) -> np.ndarray:
    # COLMAP-style extrinsics: x_cam = R_cw * x_world + t_cw
    # Camera center in world: C = -R^T t
    return -R_cw.T @ t_cw


def rotation_angle_rad(R: np.ndarray) -> float:
    # theta from trace: trace(R) = 1 + 2 cos(theta)
    c = (np.trace(R) - 1.0) / 2.0
    c = float(np.clip(c, -1.0, 1.0))
    return float(np.arccos(c))


def add_pose_metrics(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for rot, trans in tqdm(
        zip(df["rotation_matrix"].tolist(), df["translation_vector"].tolist(), strict=False),
        total=len(df),
        desc="parsing poses",
    ):
        try:
            R = parse_rotation_matrix(rot)
            t = parse_translation_vector(trans)
            C = camera_center_world(R, t)
            det = float(np.linalg.det(R))
            ortho = float(np.linalg.norm(R.T @ R - np.eye(3), ord="fro"))
            ang = rotation_angle_rad(R)
            rows.append(
                {
                    "pose_ok": True,
                    "t_x": float(t[0]),
                    "t_y": float(t[1]),
                    "t_z": float(t[2]),
                    "t_norm": float(np.linalg.norm(t)),
                    "center_x": float(C[0]),
                    "center_y": float(C[1]),
                    "center_z": float(C[2]),
                    "center_norm": float(np.linalg.norm(C)),
                    "R_det": det,
                    "R_ortho_err": ortho,
                    "rot_angle_deg": float(np.degrees(ang)),
                }
            )
        except Exception as e:
            rows.append({"pose_ok": False, "pose_error": repr(e)})

    out = df.reset_index(drop=True).copy()
    metrics = pd.DataFrame(rows)
    for c in metrics.columns:
        out[c] = metrics[c]
    return out


train_pose = add_pose_metrics(train_labels)
display(train_pose.head())
print("Pose parse ok:", int(train_pose["pose_ok"].sum()), "/", len(train_pose))


In [ ]:
px.histogram(train_pose, x="R_det", nbins=80, title="Train: det(R) distribution").show()
px.histogram(train_pose, x="R_ortho_err", nbins=80, title="Train: ||R^T R - I||_F distribution").show()
px.histogram(train_pose, x="rot_angle_deg", nbins=80, title="Train: rotation angle (deg) distribution").show()
px.histogram(train_pose, x="t_norm", nbins=80, title="Train: ||t|| distribution").show()
px.histogram(train_pose, x="center_norm", nbins=80, title="Train: ||C|| distribution").show()


In [ ]:
pose_scene_stats = (
    train_pose.groupby(["dataset", "scene"])
    .agg(
        n_images=("image", "size"),
        t_norm_median=("t_norm", "median"),
        t_norm_p90=("t_norm", lambda s: float(np.nanpercentile(s, 90))),
        center_norm_median=("center_norm", "median"),
        rot_angle_median=("rot_angle_deg", "median"),
        det_median=("R_det", "median"),
        ortho_err_p90=("R_ortho_err", lambda s: float(np.nanpercentile(s, 90))),
    )
    .reset_index()
    .sort_values("n_images", ascending=False)
)
display(pose_scene_stats.head(50))

px.scatter(
    pose_scene_stats,
    x="t_norm_median",
    y="center_norm_median",
    size="n_images",
    color="dataset",
    hover_data=["scene", "rot_angle_median", "ortho_err_p90"],
    title="Train: pose scale summary per scene",
).show()


### 3D camera center plot (per dataset/scene)

Pick a dataset and optionally a scene to visualize camera centers.

In [ ]:
PLOT_DATASET = by_dataset.iloc[0]["dataset"]
PLOT_SCENE = None  # set to a scene name to filter

subset = train_pose[train_pose["dataset"] == PLOT_DATASET]
if PLOT_SCENE is not None:
    subset = subset[subset["scene"] == PLOT_SCENE]

title = f"Camera centers: dataset={PLOT_DATASET} scene={PLOT_SCENE} (n={len(subset)})"
fig = px.scatter_3d(
    subset,
    x="center_x",
    y="center_y",
    z="center_z",
    color="scene",
    hover_name="image",
    title=title,
)
fig.update_traces(marker=dict(size=4))
fig.show()


## Optional: compare split train vs split test

Only applies when `MODE="split"` (i.e. you have held-out labels).

In [ ]:
if test_labels is None:
    print("MODE is not 'split' (no held-out labels loaded).")
else:
    test_pose = add_pose_metrics(test_labels)
    compare = pd.concat(
        [
            train_pose.assign(split="train"),
            test_pose.assign(split="test"),
        ],
        ignore_index=True,
    )
    px.histogram(compare, x="t_norm", color="split", barmode="overlay", nbins=80, title="||t||: train vs test split").show()
    px.histogram(compare, x="center_norm", color="split", barmode="overlay", nbins=80, title="||C||: train vs test split").show()


## Thresholds

Inspect `data/train_thresholds.csv` (if present).

In [ ]:
if train_thresholds is None:
    print("No thresholds CSV found.")
else:
    thr = train_thresholds.copy()
    thr_list = thr["thresholds"].astype(str).str.split(";").apply(lambda xs: [float(x) for x in xs if x])
    n = int(thr_list.map(len).max())
    thr_expanded = pd.DataFrame(thr_list.to_list(), columns=[f"thr_{i+1}" for i in range(n)])
    thr = pd.concat([thr[["dataset", "scene"]].reset_index(drop=True), thr_expanded], axis=1)
    display(thr)

    thr_long = thr.melt(id_vars=["dataset", "scene"], var_name="thr_k", value_name="threshold")
    thr_long["k"] = thr_long["thr_k"].str.replace("thr_", "").astype(int)
    px.box(thr_long, x="k", y="threshold", color="dataset", points="all", title="Threshold distributions").show()


## Next steps

- For interactive 3D inspection of a `submission.csv`, use `scripts/pose_explorer.py` (or `imc25-pose-explorer` after install).
- For retrieval clustering diagnostics, use `scripts/jigsaw_explorer.py`.
